In [1]:
from langgraph.graph import StateGraph,END
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_core.runnables import RunnablePassthrough

from langchain.tools.retriever import create_retriever_tool
from langchain.agents import AgentExecutor,create_tool_calling_agent,create_react_agent
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers.openai_functions import JsonOutputFunctionsParser,JsonKeyOutputFunctionsParser


# Number of tools

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.vectorstores import FAISS
from langchain_experimental.tools import PythonREPLTool

import functools
import os

from langchain_experimental.llms.ollama_functions import OllamaFunctions

model = OllamaFunctions(model='llama3.2')

In [2]:
from typing import TypedDict,Annotated,Sequence
import operator
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
    messages : Annotated[Sequence[BaseMessage],operator.add]


In [3]:
def create_agent(llm : ChatOllama,tools : list,message : str) -> AgentExecutor:
    prompt = ChatPromptTemplate.from_messages([
        ("system",message), MessagesPlaceholder(variable_name='messages'),MessagesPlaceholder(variable_name='agent_scratchpad')
    ])

    agent = create_tool_calling_agent(llm,tools=tools,prompt=prompt)
    agent = AgentExecutor(agent=agent,tools=tools,handle_parsing_errors=True)

    return agent

In [4]:
llm_model = ChatOllama(model='llama3.2')

rag_data_retreiver = FAISS.load_local('web_data',OllamaEmbeddings(model='llama3.2'),allow_dangerous_deserialization=True).as_retriever()
rag_data_tool = create_retriever_tool(rag_data_retreiver,"langsmith","Search for information about LangSmith. For any questions about LangSmith, you must use this tool!")

python_repl_tool = PythonREPLTool()

os.environ["TAVILY_API_KEY"] = "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"
tavily_tool = TavilySearchResults(max_results=2)

tools = [rag_data_tool,python_repl_tool,tavily_tool]



In [23]:
@tool
def rag_tool(state: list):
    ''' Use this to execute RAG. If the question is related to Langsmith, use this tool to retreive the result'''
    print()
    print()
    print("State Rag:",state)
    question = state['messages']
    question = question[0]
    print("yes rag")
    
    template = ''' Answer the question based on following context :
         {context}
      
         Question : {question}
    '''

    prompt = ChatPromptTemplate.from_messages(template)
    retreival_chain = (
        {'context':rag_data_retreiver,'question':RunnablePassthrough()}
        | prompt | ChatOllama(model='llama3.2')
    )

    response = retreival_chain.invoke(question)
    print("RAGcResponse:",response)
    return response.content

In [24]:
def agent_node(state,agent,name):
    print("State",state)
    que = state['messages']
    print("Ques:",que[0])
    result = agent.invoke(state['messages'])
    print("Result/////////",result)
    v=HumanMessage(content=result['output'],name=name)
    print("Result:",v)
    return state

In [25]:
research_agent = create_agent(llm_model,[tavily_tool],'You are a web researcher.')
research_node = functools.partial(agent_node,agent=research_agent,name='Researcher')

code_agent = create_agent(llm_model,[python_repl_tool],"You may generate safe python code to analyze the data.")
code_node = functools.partial(agent_node,agent=code_agent,name='Coder')

rag_agent = create_agent(llm_model,[rag_tool],'Use this tool when question are related to Langsmith.')
rag_node = functools.partial(agent_node,agent=rag_agent,name='RAG')

In [26]:
def supervisor(state):
    que = state['messages']
    system_prompt = (
        "You are a supervisor, your work is to distribute the task based on user message to"
        " following workers:  ['RAG', 'Coder', 'Researcher']. Following is the description of each worker"
        " RAG : RAG is used when user message is only related to Langsmith or asking about Langsmith concept."
        " Coder : Coder is used when user is asked to write the code only."
        " Researcher : He is used to only search the websites based on user messages"
        " Based on following user message : {messages},"
        " respond with the appropriate worker to act next."
        " When finished,respond with Finish."
        " Only response with worker name as next:worker_name. "
    )

    prompt = ChatPromptTemplate.from_template(template=system_prompt)

    model = ( {"messages":RunnablePassthrough()} | prompt | ChatOllama(model='llama3.2'))

    response = model.invoke(que[0])
    print("Supervisor Response:",response.content)
    return state['messages'].append(response.content)

def router(state):
    val = state['messages']
    return val[1]

In [27]:
workflow = StateGraph(AgentState)

workflow.add_node('Researcher',research_node)
workflow.add_node('Coder',code_node)
workflow.add_node('RAG',rag_node)
workflow.add_node('Supervisor',supervisor)

members = ['RAG','Researcher' , 'Coder']
for member in members:
    workflow.add_edge(member,'Supervisor')
# workflow.add_edge('Supervisor','Coder')
workflow.set_entry_point('Supervisor')

conditional_map = {k: k for k in members}
conditional_map['Finish'] = END

workflow.add_conditional_edges('Supervisor',router,conditional_map)
app = workflow.compile()

In [28]:
state = {"messages":["what is Langsmith ?"]}
app.invoke(state)

Supervisor Response: Researcher
State {'messages': ['what is Langsmith ?', 'Researcher']}
Ques: what is Langsmith ?


IndexError: list index out of range

In [11]:
type(state)

dict

By Self

In [207]:
from langgraph.graph import StateGraph,END
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_core.runnables import RunnablePassthrough

from langchain.tools.retriever import create_retriever_tool
from langchain.agents import AgentExecutor,create_tool_calling_agent,create_react_agent
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder,PromptTemplate
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers.openai_functions import JsonOutputFunctionsParser,JsonKeyOutputFunctionsParser


# Number of tools

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.vectorstores import FAISS
from langchain_experimental.tools import PythonREPLTool

import functools
import os


In [208]:
from typing import TypedDict,Annotated,Sequence
import operator
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
    messages : Annotated[Sequence[BaseMessage],operator.add]

In [209]:
from config import LLAMA_MODEL
llm_model = ChatOllama(model='llama3.2')


rag_data_retreiver = FAISS.load_local('web_data',OllamaEmbeddings(model=LLAMA_MODEL),allow_dangerous_deserialization=True).as_retriever()

python_repl_tool = PythonREPLTool()

os.environ["TAVILY_API_KEY"] = "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"
tavily_tool = TavilySearchResults(max_results=2)


In [222]:
condition = ''
def agent_node(state):
    que = state['messages']
    system_prompt = (
        "You are a supervisor, your work is to distribute the task based on user message to"
        " following workers:  ['RAG', 'Coder', 'Researcher','Finish']."
        " Distribution must be done to only one worker at a time"
        
        " Following is the description of each worker : "
        " Finish : It is used when there is no response to generate or provided answer is enough for clarification of user message."
        " RAG : RAG is used when user message is only related to Langsmith or asking about Langsmith concept."
        " Coder : Coder is used when user is asked to write the programming code only."
        " Researcher : He is used to only search the websites based on user messages"
        
        " Following is the user message : {messages},"
        " respond with the appropriate worker to act next."
        " When finished, respond with Finish."
        " Only response with worker name as next:worker_name. "

        "Also check if Answer is provided satisfactory respond with Finish."
    )

    prompt = ChatPromptTemplate.from_template(template=system_prompt)

    model = ({'messages':RunnablePassthrough()} | prompt | ChatOllama(model='llama3.2'))

    if len(que)==1:
        response = model.invoke(str(que))
    else:
        data = que[0]+ " " + que[-1]
        print("Data:",data)
        response = model.invoke(data)
    print('Supervisor Response:', response.content)
    answer = response.content
    state['messages'].append(answer)
    return state

def rag_node(state):
    que = state['messages']
    print('QuestionforRAG:', que)
    prompt = ''' Your task is to answer the Question based on following context:
                   {context}

                   Question : {question}
               '''
    template = ChatPromptTemplate.from_template(template=prompt)
    
    model = ( 
            {"context":rag_data_retreiver, "question":RunnablePassthrough()} 
            | template | ChatOllama(model='llama3.2') 
        )
    response = model.invoke(que[0])
    answer = "Answer : " + response.content
    state['messages'].append(answer)
    return state

def researcher_node(state):
    que = state['messages']
    print('Question for Researcher:',que)
    response = tavily_tool.invoke(que[0])
    print('Researcher Response:',response)
    val = response[0]
    answer = "Researcher Response : "+val['url']
    state['messages'].append(answer)
    return state

def router(state):
    value = state['messages']
    print('Condition Value:',value[-1])
    return value[-1]

In [223]:
workflow = StateGraph(AgentState)

workflow.add_node('supervisor',agent_node)
workflow.add_node('RAG',rag_node)
workflow.add_node('Researcher',researcher_node)

member = ['RAG','Researcher']
conditional_map = {}

for k in member:
    conditional_map[k]=k
    workflow.add_edge(k,'supervisor')

conditional_map['Finish'] = END
conditional_map['Coder'] = END
workflow.set_entry_point('supervisor')
workflow.add_conditional_edges('supervisor',router,conditional_map)

app=workflow.compile()

In [ ]:
state = {'messages':['what is langsmith ?']}
res=app.invoke(state)

Supervisor Response: RAG
Condition Value: RAG
QuestionforRAG: ['what is langsmith ?', 'RAG', 'what is langsmith ?', 'RAG']
Data: what is langsmith ? Answer : According to the provided context, LangSmith is a platform for building production-grade LLM (Large Language Model) applications. It allows users to closely monitor and evaluate their application, enabling them to ship quickly and with confidence.
Supervisor Response: RAG
Condition Value: RAG
QuestionforRAG: ['what is langsmith ?', 'RAG', 'what is langsmith ?', 'RAG', 'Answer : According to the provided context, LangSmith is a platform for building production-grade LLM (Large Language Model) applications. It allows users to closely monitor and evaluate their application, enabling them to ship quickly and with confidence.', 'what is langsmith ?', 'RAG', 'what is langsmith ?', 'RAG', 'Answer : According to the provided context, LangSmith is a platform for building production-grade LLM (Large Language Model) applications. It allows u

In [205]:
res.get('messages')[1]

"Answer : Based on the context provided, LangSmith appears to be a platform for building production-grade LLM (Large Language Model) applications. It allows users to closely monitor and evaluate their application, enabling them to ship quickly and with confidence. With LangSmith, users can gain visibility into LLM calls and other parts of their application's logic, compare results across models, prompts, and architectures to identify what works best, and quickly refine prompts to achieve more accurate and reliable results."